# DeepONet vs. PINN: 1-D Heat Equation

This notebook trains **two** neural network solvers for the 1-D heat (diffusion) equation

$$ \frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}, \qquad x \in [0,1],\; t \in [0,1] $$

with homogeneous Dirichlet boundary conditions $u(0,t)=u(1,t)=0$ and initial condition $u(x,0)=u_0(x)$.

- **PINN (Physics-Informed Neural Network)** learns a single function $u(x,t)$ that solves the PDE for **one fixed** initial condition $u_0$. If $u_0$ changes, the network must be **retrained from scratch**.
- **DeepONet** learns the *solution operator* $G: u_0(\cdot) \mapsto u(\cdot,\cdot)$. Once trained on a distribution of initial conditions, it can predict the solution for a **brand-new** initial condition **instantly**, with no retraining.

Because the heat equation with these boundary conditions has a closed-form sine-series solution, we can generate exact training/reference data for both methods and measure errors precisely. The final section is the key comparison: we test both models on an initial condition that was **never seen by the PINN**, to make the generalization gap obvious.

> Runtime: a few minutes on a Colab CPU runtime (GPU not required, but will also work).


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import time
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 1. Problem setup and analytical (reference) solution

For $u(x,0) = u_0(x) = \sum_{k=1}^{K} a_k \sin(k\pi x)$, the exact solution of the heat equation with $u(0,t)=u(1,t)=0$ is

$$ u(x,t) = \sum_{k=1}^{K} a_k \, e^{-\alpha (k\pi)^2 t} \sin(k\pi x). $$

We use this closed form to (a) build a training set of many random initial conditions for DeepONet, and (b) compute exact errors for both models.


In [ ]:
ALPHA = 0.02      # thermal diffusivity
K_MODES = 4       # number of sine modes used to build random initial conditions
M_SENSORS = 50    # number of sensor points used to describe u0(x) for DeepONet's branch net

x_sensors = np.linspace(0.0, 1.0, M_SENSORS)  # fixed sensor grid for the branch input


def ic_from_coeffs(a, x):
    """u0(x) = sum_k a_k sin(k pi x)"""
    x = np.asarray(x)
    k = np.arange(1, len(a) + 1)
    return np.sin(np.outer(x, k) * np.pi) @ a


def exact_solution(a, x, t):
    """u(x,t) for the sine-series initial condition with coefficients a."""
    x = np.asarray(x)
    t = np.asarray(t)
    k = np.arange(1, len(a) + 1)
    decay = np.exp(-ALPHA * (k * np.pi) ** 2 * t[:, None])          # (Nt, K)
    spatial = np.sin(np.outer(x, k) * np.pi)                        # (Nx, K)
    # broadcast for paired (x,t) query points (same length arrays)
    return np.sum(decay * spatial * a[None, :], axis=1)


def random_coeffs(k_modes=K_MODES, rng=np.random):
    return rng.uniform(-1.0, 1.0, size=k_modes)


## 2. DeepONet: learning the solution operator

**Training data.** We sample many random initial conditions $u_0^{(i)}$ (random sine-mode coefficients). For each one we:
- evaluate $u_0^{(i)}$ at the fixed sensor points -> **branch input**,
- pick random query points $(x,t)$ and evaluate the exact solution -> **trunk input** + **target**.

**Model.** DeepONet = branch net $b(u_0) \in \mathbb{R}^p$ + trunk net $\tau(x,t) \in \mathbb{R}^p$, combined as
$$ G(u_0)(x,t) \approx \sum_{i=1}^{p} b_i(u_0)\, \tau_i(x,t) + c. $$


In [ ]:
N_FUNCS = 500        # number of random initial conditions in the training set
N_PTS_PER_FUNC = 50  # random (x,t) query points sampled per initial condition

branch_inputs = np.zeros((N_FUNCS * N_PTS_PER_FUNC, M_SENSORS), dtype=np.float32)
trunk_inputs = np.zeros((N_FUNCS * N_PTS_PER_FUNC, 2), dtype=np.float32)
targets = np.zeros((N_FUNCS * N_PTS_PER_FUNC, 1), dtype=np.float32)

rng = np.random.RandomState(42)
row = 0
for i in range(N_FUNCS):
    a = random_coeffs(rng=rng)
    u0_vals = ic_from_coeffs(a, x_sensors)

    xq = rng.uniform(0.0, 1.0, size=N_PTS_PER_FUNC)
    tq = rng.uniform(0.0, 1.0, size=N_PTS_PER_FUNC)
    uq = exact_solution(a, xq, tq)

    branch_inputs[row:row + N_PTS_PER_FUNC] = u0_vals
    trunk_inputs[row:row + N_PTS_PER_FUNC, 0] = xq
    trunk_inputs[row:row + N_PTS_PER_FUNC, 1] = tq
    targets[row:row + N_PTS_PER_FUNC, 0] = uq
    row += N_PTS_PER_FUNC

branch_inputs_t = torch.tensor(branch_inputs, device=device)
trunk_inputs_t = torch.tensor(trunk_inputs, device=device)
targets_t = torch.tensor(targets, device=device)

print("DeepONet training set:", branch_inputs_t.shape, trunk_inputs_t.shape, targets_t.shape)


In [ ]:
def mlp(sizes, activation=nn.ReLU, final_activation=False):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1]))
        if i < len(sizes) - 2 or final_activation:
            layers.append(activation())
    return nn.Sequential(*layers)


class DeepONet(nn.Module):
    def __init__(self, m_sensors, p=64, hidden=128):
        super().__init__()
        self.branch = mlp([m_sensors, hidden, hidden, p])
        self.trunk = mlp([2, hidden, hidden, p], final_activation=True)
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, u0, xt):
        b = self.branch(u0)
        t = self.trunk(xt)
        return torch.sum(b * t, dim=-1, keepdim=True) + self.bias


deeponet = DeepONet(M_SENSORS).to(device)
optimizer = torch.optim.Adam(deeponet.parameters(), lr=1e-3)

N_ITERS = 3000
BATCH = 256
n_samples = branch_inputs_t.shape[0]
loss_history = []

t0 = time.time()
for it in range(N_ITERS):
    idx = torch.randint(0, n_samples, (BATCH,), device=device)
    pred = deeponet(branch_inputs_t[idx], trunk_inputs_t[idx])
    loss = torch.mean((pred - targets_t[idx]) ** 2)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    if (it + 1) % 500 == 0:
        print(f"[DeepONet] iter {it+1:5d}  MSE loss = {loss.item():.3e}")

deeponet_train_time = time.time() - t0
print(f"DeepONet training time: {deeponet_train_time:.1f}s")


In [ ]:
plt.figure(figsize=(5, 3))
plt.semilogy(loss_history)
plt.xlabel("iteration")
plt.ylabel("MSE loss")
plt.title("DeepONet training loss")
plt.tight_layout()
plt.show()


## 3. PINN: solving the PDE for one fixed initial condition

The PINN is a plain MLP $u_\theta(x,t)$ trained by minimizing the PDE residual (via automatic differentiation) plus the boundary/initial condition mismatch. **It only ever sees one initial condition** — it has no notion of "different problems", unlike DeepONet's branch net.


In [ ]:
# The single initial condition the PINN is trained for.
a_train = np.array([1.0, 0.0, 0.5, 0.0])   # u0(x) = sin(pi x) + 0.5 sin(3 pi x)


class PINN(nn.Module):
    def __init__(self, hidden=50, layers=4):
        super().__init__()
        sizes = [2] + [hidden] * layers + [1]
        modules = []
        for i in range(len(sizes) - 1):
            modules.append(nn.Linear(sizes[i], sizes[i + 1]))
            if i < len(sizes) - 2:
                modules.append(nn.Tanh())
        self.net = nn.Sequential(*modules)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


def pde_residual(model, x, t, alpha):
    x = x.clone().requires_grad_(True)
    t = t.clone().requires_grad_(True)
    u = model(x, t)
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    return u_t - alpha * u_xx


def train_pinn(a_coeffs, n_iters=8000, lr=1e-3, n_colloc=2000, n_bc=200, n_ic=200, seed=0, verbose=True):
    torch.manual_seed(seed)
    model = PINN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    rng_local = np.random.RandomState(seed)
    history = []

    t0 = time.time()
    for it in range(n_iters):
        # interior collocation points -> PDE residual loss
        x_c = torch.tensor(rng_local.uniform(0, 1, size=(n_colloc, 1)), dtype=torch.float32, device=device)
        t_c = torch.tensor(rng_local.uniform(0, 1, size=(n_colloc, 1)), dtype=torch.float32, device=device)
        res = pde_residual(model, x_c, t_c, ALPHA)
        loss_pde = torch.mean(res ** 2)

        # boundary points x=0 and x=1 -> u = 0
        t_b = torch.tensor(rng_local.uniform(0, 1, size=(n_bc, 1)), dtype=torch.float32, device=device)
        x_b0 = torch.zeros_like(t_b)
        x_b1 = torch.ones_like(t_b)
        loss_bc = torch.mean(model(x_b0, t_b) ** 2) + torch.mean(model(x_b1, t_b) ** 2)

        # initial points t=0 -> u = u0(x)
        x_i = torch.tensor(rng_local.uniform(0, 1, size=(n_ic, 1)), dtype=torch.float32, device=device)
        t_i = torch.zeros_like(x_i)
        u0_target = torch.tensor(
            ic_from_coeffs(a_coeffs, x_i.detach().cpu().numpy().flatten())[:, None],
            dtype=torch.float32, device=device,
        )
        loss_ic = torch.mean((model(x_i, t_i) - u0_target) ** 2)

        loss = loss_pde + loss_bc + loss_ic

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append(loss.item())
        if verbose and (it + 1) % 1000 == 0:
            print(f"[PINN] iter {it+1:5d}  total={loss.item():.3e}  "
                  f"pde={loss_pde.item():.3e}  bc={loss_bc.item():.3e}  ic={loss_ic.item():.3e}")

    train_time = time.time() - t0
    return model, history, train_time


pinn_model, pinn_history, pinn_train_time = train_pinn(a_train)
print(f"PINN training time: {pinn_train_time:.1f}s")


In [ ]:
plt.figure(figsize=(5, 3))
plt.semilogy(pinn_history)
plt.xlabel("iteration")
plt.ylabel("total loss")
plt.title("PINN training loss")
plt.tight_layout()
plt.show()


## 4. Sanity check: both models on the initial condition the PINN was trained on

Here both models should do well: the PINN was trained specifically for this initial condition, and this IC lies inside the distribution DeepONet was trained on.


In [ ]:
def eval_grid(a_coeffs, nx=100, nt=100):
    xs = np.linspace(0, 1, nx)
    ts = np.linspace(0, 1, nt)
    Xg, Tg = np.meshgrid(xs, ts, indexing="ij")
    x_flat = Xg.flatten()
    t_flat = Tg.flatten()
    u_exact = exact_solution(a_coeffs, x_flat, t_flat).reshape(nx, nt)
    return xs, ts, Xg, Tg, u_exact


def deeponet_predict(a_coeffs, x_flat, t_flat):
    u0_vals = ic_from_coeffs(a_coeffs, x_sensors).astype(np.float32)
    u0_batch = np.tile(u0_vals, (len(x_flat), 1))
    xt_batch = np.stack([x_flat, t_flat], axis=1).astype(np.float32)
    with torch.no_grad():
        pred = deeponet(torch.tensor(u0_batch, device=device), torch.tensor(xt_batch, device=device))
    return pred.cpu().numpy().flatten()


def pinn_predict(model, x_flat, t_flat):
    with torch.no_grad():
        x_t = torch.tensor(x_flat[:, None], dtype=torch.float32, device=device)
        t_t = torch.tensor(t_flat[:, None], dtype=torch.float32, device=device)
        pred = model(x_t, t_t)
    return pred.cpu().numpy().flatten()


def relative_l2_error(pred, exact):
    return np.linalg.norm(pred - exact) / np.linalg.norm(exact)


def plot_comparison(title, xs, ts, Xg, u_exact, u_deeponet, u_pinn):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2), sharey=True)
    for ax, field, name in zip(
        axes, [u_exact, u_deeponet, u_pinn], ["Exact", "DeepONet", "PINN"]
    ):
        im = ax.pcolormesh(ts, xs, field, shading="auto", cmap="viridis")
        ax.set_title(name)
        ax.set_xlabel("t")
    axes[0].set_ylabel("x")
    fig.suptitle(title)
    fig.colorbar(im, ax=axes, shrink=0.85)
    plt.show()


xs, ts, Xg, Tg, u_exact_train_ic = eval_grid(a_train)
u_deeponet_train_ic = deeponet_predict(a_train, Xg.flatten(), Tg.flatten()).reshape(Xg.shape)
u_pinn_train_ic = pinn_predict(pinn_model, Xg.flatten(), Tg.flatten()).reshape(Xg.shape)

plot_comparison("Trained IC: u0(x) = sin(pi x) + 0.5 sin(3 pi x)",
                 xs, ts, Xg, u_exact_train_ic, u_deeponet_train_ic, u_pinn_train_ic)

print(f"Relative L2 error | DeepONet: {relative_l2_error(u_deeponet_train_ic, u_exact_train_ic):.4e}"
      f"   PINN: {relative_l2_error(u_pinn_train_ic, u_exact_train_ic):.4e}")


## 5. The key comparison: a brand-new initial condition

Now we evaluate both models on a **new initial condition that the PINN never saw during training**. DeepONet was trained on a *distribution* of random initial conditions, so a new one is just another sample from that distribution — no retraining needed. The PINN, however, is a solution for one fixed problem; asked about a different initial condition, it has **no way to know about it** and its prediction is simply wrong.

To make the comparison fair, we also **retrain a fresh PINN from scratch** for the new initial condition, and time it — this is the cost DeepONet avoids entirely.


In [ ]:
a_new = np.array([0.0, 1.0, 0.0, -0.3])  # u0(x) = sin(2 pi x) - 0.3 sin(4 pi x); unseen by the trained PINN

xs, ts, Xg, Tg, u_exact_new = eval_grid(a_new)

# (a) DeepONet: zero-shot, no retraining
t0 = time.time()
u_deeponet_new = deeponet_predict(a_new, Xg.flatten(), Tg.flatten()).reshape(Xg.shape)
deeponet_inference_time = time.time() - t0

# (b) Stale PINN: reuse the network trained for a_train, unchanged
u_pinn_stale_new = pinn_predict(pinn_model, Xg.flatten(), Tg.flatten()).reshape(Xg.shape)

# (c) Retrained PINN: train a brand-new network specifically for a_new
pinn_model_new, pinn_history_new, pinn_retrain_time = train_pinn(a_new, seed=1)
u_pinn_retrained_new = pinn_predict(pinn_model_new, Xg.flatten(), Tg.flatten()).reshape(Xg.shape)

plot_comparison("Unseen IC: u0(x) = sin(2 pi x) - 0.3 sin(4 pi x)  (DeepONet: zero-shot | PINN: reused, untrained for this IC)",
                 xs, ts, Xg, u_exact_new, u_deeponet_new, u_pinn_stale_new)

plot_comparison("Same unseen IC, after retraining a fresh PINN from scratch",
                 xs, ts, Xg, u_exact_new, u_deeponet_new, u_pinn_retrained_new)

err_deeponet = relative_l2_error(u_deeponet_new, u_exact_new)
err_pinn_stale = relative_l2_error(u_pinn_stale_new, u_exact_new)
err_pinn_retrained = relative_l2_error(u_pinn_retrained_new, u_exact_new)

print("=== New, unseen initial condition ===")
print(f"DeepONet (zero-shot)         : rel. L2 error = {err_deeponet:.4e}   inference time = {deeponet_inference_time*1000:.1f} ms")
print(f"PINN (reused, not retrained) : rel. L2 error = {err_pinn_stale:.4e}   inference time ~ 0 ms (but WRONG problem)")
print(f"PINN (retrained from scratch): rel. L2 error = {err_pinn_retrained:.4e}   retraining time = {pinn_retrain_time:.1f} s")


## 6. Takeaways

| | PINN | DeepONet |
|---|---|---|
| What it learns | One solution $u(x,t)$ for **one fixed** initial condition | The operator $u_0(\cdot) \mapsto u(\cdot,\cdot)$ across a **distribution** of initial conditions |
| New initial condition | Must **retrain from scratch** (seconds-to-minutes here; much longer for harder PDEs) | **Instant** forward pass (milliseconds), no retraining |
| Training data needed | None (just the PDE + IC/BC) | Many example (initial condition -> solution) pairs |
| Accuracy on a seen problem | Very good (network optimized for exactly that problem) | Good, provided the new IC is within the training distribution |

This is the essential trade-off: PINNs are a mesh-free PDE **solver** for a single problem instance, while DeepONet is a **surrogate for the solution operator** that amortizes training cost over many instances, at the price of needing a training dataset of solved examples up front. The generalization experiment above (Section 5) is exactly where this difference becomes visible: DeepONet's error stays low with zero extra cost, while the PINN trained on the old IC is simply solving the wrong problem until it is retrained.
